## 5.7. Erweiterte NLP- und Deep-Learning-Modelle

Neben den klassischen scikit-learn-Modellen werden zwei weitere Modellfamilien vorbereitet: `fastText` und ein deutschsprachiges bzw. multilingual einsetzbares BERT-Modell. Beide Modelle benötigen eine andere Datenrepräsentation als die bisherige `ColumnTransformer`-Pipeline. Deshalb werden sie hier nicht direkt in die bestehende Cross-Validation integriert, sondern als optionale Erweiterung vorbereitet.

`fastText` erwartet eine Textdatei mit Labels im Format `__label__Klasse Text`. BERT arbeitet dagegen mit einem Tokenizer, der den zusammengeführten Text in Token-IDs übersetzt. Die folgenden Zellen definieren daher die jeweils passende Eingaberepräsentation. Die eigentliche Modellschätzung ist bewusst deaktiviert und kann bei ausreichender Rechenzeit eingeschaltet werden.


### 5.7.1. Gemeinsame Textrepräsentation für NLP-Modelle

Für beide Modelle werden die relevanten Modellspalten zu einem einzigen Textfeld zusammengeführt. Dabei bleiben die Spaltennamen als Feldmarker erhalten. Dadurch kann das Modell unterscheiden, ob ein Begriff aus dem Namen, dem Zweck, der Anschrift oder einer kategorialen Angabe stammt.

Als Ausgangspunkt wird das beste Szenario aus den vorherigen Repräsentationsexperimenten verwendet. Falls dieses Objekt bei einem isolierten Notebook-Lauf noch nicht existiert, wird eine robuste Standardauswahl aus Text- und kategorialen Spalten verwendet.


In [2]:
from pathlib import Path
import os
import sys

project_root = globals().get(
    "project_root",
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
)

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from model_evaluation import combine_columns_as_text, make_fasttext_label

# Hugging-Face-Konfiguration: Der Token wird nicht im Notebook gespeichert.
# Falls HF_TOKEN bereits als Umgebungsvariable oder Notebook-Variable existiert,
# wird er fuer Downloads vom Hugging Face Hub verwendet.
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
hf_token = globals().get("HF_TOKEN", os.environ.get("HF_TOKEN"))
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
else:
    hf_token = None

import time

import pandas as pd
from IPython.display import display
from joblib import dump

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder

model_output_dir = globals().get(
    "model_output_dir",
    project_root / "models",
)
model_output_dir.mkdir(parents=True, exist_ok=True)

try:
    import mlflow

    mlflow_tracking_dir = project_root / "mlruns"
    mlflow.set_tracking_uri(mlflow_tracking_dir.as_uri())
    mlflow.set_experiment("politikbereich_classifier")
except ImportError:
    pass

RUN_FASTTEXT_BASELINE = globals().get("RUN_FASTTEXT_BASELINE", True)
RUN_BERT_BASELINE = globals().get("RUN_BERT_BASELINE", True)

if "train_df" not in globals():
    train_df = pd.read_csv(
        project_root / "data" / "processed" / "train_data.csv",
        sep=";",
        index_col=0,
        encoding="utf-8",
    )

if "target_column" not in globals():
    target_column = "politikbereich"

if "text_features" not in globals():
    text_features = [
        "name_standardised",
        "geber_standardised",
        "anschrift_standardised",
        "zweck_standardised",
    ]

if "numeric_features" not in globals():
    numeric_features = [
        "betrag",
    ]

if "categorical_features" not in globals():
    categorical_features = [
        "art_standardised",
        "jahr",
    ]

if "feature_columns" not in globals():
    feature_columns = text_features + numeric_features + categorical_features

if "X_train" not in globals() or "y_train" not in globals():
    available_feature_columns = [
        column
        for column in feature_columns
        if column in train_df.columns
    ]

    X_train = train_df[available_feature_columns].copy()
    y_train = train_df[target_column].copy()

bert_model_checkpoint = globals().get(
    "bert_model_checkpoint",
    "deepset/gbert-base",
)

if "n_splits" not in globals():
    min_class_count = y_train.value_counts().min()
    n_splits = min(4, min_class_count)

if "cv" not in globals():
    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42,
    )

print("Trainingsdaten:", X_train.shape)
print("Zielvariable:", target_column)

if "best_scenario_config" in globals():
    extended_nlp_columns = (
        best_scenario_config["text_features"]
        + best_scenario_config["categorical_features"]
        + best_scenario_config["numeric_features"]
    )
else:
    extended_nlp_columns = (
        text_features
        + categorical_features
    )

# Für reine NLP-Modelle werden numerische Spalten nur dann verwendet,
# wenn sie im besten Szenario explizit enthalten sind.
extended_nlp_columns = [
    column
    for column in extended_nlp_columns
    if column in X_train.columns
]

X_train_nlp = X_train.copy()
X_train_nlp["model_text"] = combine_columns_as_text(
    X_train_nlp,
    extended_nlp_columns,
)

print("Verwendete Spalten für NLP-Modelle:")
display(
    pd.DataFrame(
        {"Trainingsspalte": extended_nlp_columns}
    )
)

display(
    X_train_nlp[["model_text"]].head(3)
)


Trainingsdaten: (45920, 7)
Zielvariable: politikbereich
Verwendete Spalten für NLP-Modelle:


,Trainingsspalte
0,name_standardised
1,geber_standardised
2,anschrift_standardised
3,zweck_standardised
4,art_standardised
5,jahr


,model_text
id,
33655,name: cashmere radio e. v. geber: senatsverwal...
105910,name: merantix labs gmbh geber: senatsverwaltu...
67465,name: georg kolbe-stiftung geber: senatsverwal...


### 5.7.2. fastText vorbereiten

`fastText` ist für Textklassifikation sehr effizient, weil es Wort- und n-Gramm-Informationen direkt aus einer einfachen Textdatei lernt. Es passt jedoch nicht direkt in die bestehende scikit-learn-Pipeline. Deshalb wird hier zuerst eine fastText-kompatible Trainingsdatei erzeugt.

Die Trainingszelle ist standardmäßig deaktiviert. Wenn `fasttext` installiert ist und das Modell tatsächlich trainiert werden soll, kann `train_fasttext_model` auf `True` gesetzt werden.


In [2]:
fasttext_training_data = pd.DataFrame(
    {
        "label": y_train.map(make_fasttext_label),
        "text": X_train_nlp["model_text"],
    }
)

fasttext_training_path = (
    project_root
    / "data"
    / "processed"
    / "fasttext_train.txt"
)

fasttext_training_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

fasttext_lines = (
    fasttext_training_data["label"]
    + " "
    + fasttext_training_data["text"]
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
)

fasttext_training_path.write_text(
    "\n".join(fasttext_lines.tolist()),
    encoding="utf-8",
)

print("fastText-Trainingsdatei gespeichert:", fasttext_training_path)
display(fasttext_training_data.head(3))

train_fasttext_model = True

if train_fasttext_model:
    try:
        import fasttext

        fasttext_model = fasttext.train_supervised(
            input=str(fasttext_training_path),
            epoch=10,
            lr=0.5,
            wordNgrams=2,
            loss="softmax",
        )

        fasttext_model_path = (
            model_output_dir
            / "fasttext_politikbereich.bin"
        )
        fasttext_model.save_model(str(fasttext_model_path))

        print("fastText-Modell gespeichert:", fasttext_model_path)

    except ImportError:
        print(
            "fastText ist nicht installiert. "
            "Bei Bedarf kann es separat installiert werden."
        )
else:
    print(
        "fastText-Training ist deaktiviert. "
        "Setze train_fasttext_model = True, um es auszuführen."
    )


fastText-Trainingsdatei gespeichert: d:\toydev\schwarz-test\data\processed\fasttext_train.txt


,label,text
id,,
33655,__label__Kultur,name: cashmere radio e. v. geber: senatsverwal...
105910,__label__Wirtschaft,name: merantix labs gmbh geber: senatsverwaltu...
67465,__label__Kultur,name: georg kolbe-stiftung geber: senatsverwal...


fastText-Modell gespeichert: d:\toydev\schwarz-test\models\fasttext_politikbereich.bin


### 5.7.3. German BERT / multilingual BERT vorbereiten

BERT-Modelle verwenden keinen TF-IDF-Vektorizer, sondern einen Tokenizer. Dieser zerlegt den zusammengeführten Text in Token-IDs, Attention Masks und weitere Eingaben für ein Transformer-Modell. Dadurch kann semantischer Kontext besser berücksichtigt werden, allerdings ist das Fine-Tuning deutlich rechenintensiver als die klassischen Baselines.

Hier wird nur die benötigte Daten- und Encoderstruktur vorbereitet. Das eigentliche Fine-Tuning bleibt deaktiviert, damit der Baseline-Notebook-Lauf weiterhin leicht reproduzierbar bleibt.


In [3]:
bert_label_encoder = LabelEncoder()
y_train_bert = bert_label_encoder.fit_transform(y_train)

bert_label_mapping = pd.DataFrame(
    {
        "label_id": range(len(bert_label_encoder.classes_)),
        "politikbereich": bert_label_encoder.classes_,
    }
)

bert_label_mapping_path = (
    project_root
    / "data"
    / "processed"
    / "bert_label_mapping.csv"
)

bert_label_mapping.to_csv(
    bert_label_mapping_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

print("BERT-Labelmapping gespeichert:", bert_label_mapping_path)
display(bert_label_mapping.head())

prepare_bert_tokenizer = False
bert_model_checkpoint = "deepset/gbert-base"

if prepare_bert_tokenizer:
    try:
        import torch
        from transformers import AutoTokenizer

        bert_tokenizer = AutoTokenizer.from_pretrained(
            bert_model_checkpoint,
            token=hf_token,
            use_fast=False,
        )

        bert_encoded_sample = bert_tokenizer(
            X_train_nlp["model_text"].head(3).tolist(),
            truncation=True,
            padding=True,
            max_length=256,
        )

        class PolitikbereichTextDataset(torch.utils.data.Dataset):
            """Dataset-Klasse für BERT-Fine-Tuning."""

            def __init__(
                self,
                texts,
                labels,
                tokenizer,
                max_length=256,
            ):
                self.texts = list(texts)
                self.labels = list(labels)
                self.tokenizer = tokenizer
                self.max_length = max_length

            def __len__(self):
                return len(self.texts)

            def __getitem__(self, index):
                encoded = self.tokenizer(
                    self.texts[index],
                    truncation=True,
                    padding="max_length",
                    max_length=self.max_length,
                    return_tensors="pt",
                )

                item = {
                    key: value.squeeze(0)
                    for key, value in encoded.items()
                }
                item["labels"] = torch.tensor(
                    self.labels[index],
                    dtype=torch.long,
                )

                return item

        bert_train_dataset = PolitikbereichTextDataset(
            X_train_nlp["model_text"],
            y_train_bert,
            bert_tokenizer,
            max_length=256,
        )

        print(
            "BERT-Tokenizer vorbereitet:",
            bert_model_checkpoint,
        )
        print(
            "Beispiel-Keys:",
            list(bert_encoded_sample.keys()),
        )

    except ImportError:
        print(
            "transformers und/oder torch sind nicht installiert. "
            "Für BERT-Fine-Tuning müssen diese Pakete separat installiert werden."
        )
else:
    print(
        "BERT-Tokenisierung ist deaktiviert. "
        "Setze prepare_bert_tokenizer = True, um sie auszuführen."
    )


BERT-Labelmapping gespeichert: d:\toydev\schwarz-test\data\processed\bert_label_mapping.csv


,label_id,politikbereich
0,0,Antidiskriminierung
1,1,Arbeit
2,2,"Bauen, Wohnen"
3,3,Berlin-Image
4,4,Bildung


BERT-Tokenisierung ist deaktiviert. Setze prepare_bert_tokenizer = True, um sie auszuführen.


### 5.7.4. fastText trainieren, auswerten und speichern

Für fastText wird aus den Trainingsdaten ein kleiner interner Validierungssplit erzeugt. Dadurch kann das Modell direkt beurteilt werden, ohne den finalen Testdatensatz zu verwenden. Das Modell wird anschließend als `.bin`-Datei gespeichert. Zusätzlich werden Labelmapping, Summary und klassenbezogener Report abgelegt.


In [4]:
if RUN_FASTTEXT_BASELINE:
    fasttext_model_dir = model_output_dir / "fasttext"
    fasttext_model_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    fasttext_label_mapping = (
        pd.DataFrame(
            {
                "politikbereich": sorted(y_train.unique()),
            }
        )
        .assign(
            fasttext_label=lambda dataframe: dataframe["politikbereich"].map(
                make_fasttext_label
            )
        )
    )

    if fasttext_label_mapping["fasttext_label"].duplicated().any():
        duplicated_labels = fasttext_label_mapping.loc[
            fasttext_label_mapping["fasttext_label"].duplicated(keep=False)
        ]
        raise ValueError(
            "Nicht eindeutige fastText-Labels gefunden. "
            "Bitte Label-Erzeugung prüfen."
        )

    fasttext_label_to_class = dict(
        zip(
            fasttext_label_mapping["fasttext_label"],
            fasttext_label_mapping["politikbereich"],
        )
    )

    fasttext_label_mapping_path = (
        fasttext_model_dir
        / "fasttext_label_mapping.csv"
    )
    fasttext_label_mapping.to_csv(
        fasttext_label_mapping_path,
        index=False,
        sep=";",
        encoding="utf-8",
    )

    def write_fasttext_file(
        path,
        texts,
        labels,
    ):
        lines = (
            labels.map(make_fasttext_label)
            + " "
            + texts
            .fillna("")
            .astype(str)
            .str.replace("\n", " ", regex=False)
            .str.replace("\r", " ", regex=False)
        )

        path.write_text(
            "\n".join(lines.tolist()),
            encoding="utf-8",
        )

    def predict_fasttext_labels(
        model,
        texts,
        index,
    ):
        predicted_labels, _ = model.predict(
            texts.tolist(),
            k=1,
        )

        return pd.Series(
            [
                fasttext_label_to_class.get(label_group[0], label_group[0])
                if len(label_group) > 0
                else None
                for label_group in predicted_labels
            ],
            index=index,
        )

    try:
        import fasttext

        fasttext_params = {
            "epoch": 10,
            "lr": 0.5,
            "wordNgrams": 2,
            "minn": 3,
            "maxn": 6,
            "loss": "softmax",
            "verbose": 2,
        }

        fold_rows = []
        out_of_fold_predictions = []
        out_of_fold_true_values = []

        for fold_number, (train_index, valid_index) in enumerate(
            cv.split(X_train_nlp["model_text"], y_train),
            start=1,
        ):
            fold_train_texts = X_train_nlp["model_text"].iloc[train_index]
            fold_valid_texts = X_train_nlp["model_text"].iloc[valid_index]
            fold_y_train = y_train.iloc[train_index]
            fold_y_valid = y_train.iloc[valid_index]

            fold_train_path = (
                fasttext_model_dir
                / f"train_fold_{fold_number}.txt"
            )
            write_fasttext_file(
                fold_train_path,
                fold_train_texts,
                fold_y_train,
            )

            fold_start_time = time.perf_counter()
            fold_model = fasttext.train_supervised(
                input=str(fold_train_path),
                **fasttext_params,
            )
            fold_fit_time = time.perf_counter() - fold_start_time

            fold_y_pred = predict_fasttext_labels(
                fold_model,
                fold_valid_texts,
                fold_y_valid.index,
            )

            fold_rows.append(
                {
                    "fold": fold_number,
                    "fit_time_seconds": round(fold_fit_time, 2),
                    "val_accuracy": accuracy_score(
                        fold_y_valid,
                        fold_y_pred,
                    ),
                    "val_macro_f1": f1_score(
                        fold_y_valid,
                        fold_y_pred,
                        average="macro",
                        zero_division=0,
                    ),
                    "val_weighted_f1": f1_score(
                        fold_y_valid,
                        fold_y_pred,
                        average="weighted",
                        zero_division=0,
                    ),
                    "val_balanced_accuracy": balanced_accuracy_score(
                        fold_y_valid,
                        fold_y_pred,
                    ),
                }
            )

            out_of_fold_predictions.append(fold_y_pred)
            out_of_fold_true_values.append(fold_y_valid)

        fasttext_fold_results = pd.DataFrame(fold_rows)

        fasttext_full_train_path = (
            fasttext_model_dir
            / "full_train.txt"
        )
        write_fasttext_file(
            fasttext_full_train_path,
            X_train_nlp["model_text"],
            y_train,
        )

        final_fasttext_start_time = time.perf_counter()
        fitted_fasttext_model = fasttext.train_supervised(
            input=str(fasttext_full_train_path),
            **fasttext_params,
        )
        final_training_time = (
            time.perf_counter()
            - final_fasttext_start_time
        )

        fasttext_model_path = (
            fasttext_model_dir
            / "fasttext_politikbereich.bin"
        )
        fitted_fasttext_model.save_model(str(fasttext_model_path))

        fasttext_summary = pd.DataFrame(
            [
                {
                    "model": "fastText",
                    "CV_Folds": n_splits,
                    "val_accuracy_mean": fasttext_fold_results[
                        "val_accuracy"
                    ].mean(),
                    "val_accuracy_std": fasttext_fold_results[
                        "val_accuracy"
                    ].std(ddof=0),
                    "val_macro_f1_mean": fasttext_fold_results[
                        "val_macro_f1"
                    ].mean(),
                    "val_macro_f1_std": fasttext_fold_results[
                        "val_macro_f1"
                    ].std(ddof=0),
                    "val_weighted_f1_mean": fasttext_fold_results[
                        "val_weighted_f1"
                    ].mean(),
                    "val_weighted_f1_std": fasttext_fold_results[
                        "val_weighted_f1"
                    ].std(ddof=0),
                    "val_balanced_accuracy_mean": fasttext_fold_results[
                        "val_balanced_accuracy"
                    ].mean(),
                    "val_balanced_accuracy_std": fasttext_fold_results[
                        "val_balanced_accuracy"
                    ].std(ddof=0),
                    "fit_time_total_seconds": fasttext_fold_results[
                        "fit_time_seconds"
                    ].sum(),
                    "final_training_time_seconds": round(
                        final_training_time,
                        2,
                    ),
                    "model_path": str(fasttext_model_path),
                    "validation_strategy": "shared_stratified_4_fold_cv",
                }
            ]
        )

        fasttext_y_valid_all = pd.concat(
            out_of_fold_true_values
        ).sort_index()
        fasttext_y_pred_all = pd.concat(
            out_of_fold_predictions
        ).loc[fasttext_y_valid_all.index]

        fasttext_report = pd.DataFrame(
            classification_report(
                fasttext_y_valid_all,
                fasttext_y_pred_all,
                output_dict=True,
                zero_division=0,
            )
        ).transpose()

        fasttext_summary_path = fasttext_model_dir / "fasttext_summary.csv"
        fasttext_fold_results_path = (
            fasttext_model_dir
            / "fasttext_cv_fold_results.csv"
        )
        fasttext_report_path = (
            fasttext_model_dir
            / "fasttext_classification_report.csv"
        )

        fasttext_summary.to_csv(
            fasttext_summary_path,
            index=False,
            sep=";",
            encoding="utf-8",
        )
        fasttext_fold_results.to_csv(
            fasttext_fold_results_path,
            index=False,
            sep=";",
            encoding="utf-8",
        )
        fasttext_report.to_csv(
            fasttext_report_path,
            sep=";",
            encoding="utf-8",
        )

        if "mlflow" in globals():
            with mlflow.start_run(
                run_name="fastText shared CV",
                nested=True,
            ):
                mlflow.log_param("model", "fastText")
                mlflow.log_param(
                    "validation_strategy",
                    "shared_stratified_4_fold_cv",
                )
                for parameter_name, parameter_value in fasttext_params.items():
                    mlflow.log_param(parameter_name, parameter_value)

                for metric_name, metric_value in fasttext_summary.iloc[0].items():
                    if isinstance(metric_value, (int, float)):
                        mlflow.log_metric(metric_name, metric_value)

                mlflow.log_artifact(str(fasttext_model_path))
                mlflow.log_artifact(str(fasttext_summary_path))
                mlflow.log_artifact(str(fasttext_fold_results_path))
                mlflow.log_artifact(str(fasttext_report_path))

        print("fastText-Modell gespeichert:", fasttext_model_path)
        print("fastText-Summary gespeichert:", fasttext_summary_path)
        display(fasttext_summary)
        display(fasttext_fold_results)
        display(fasttext_report.head(10))

    except ImportError:
        print(
            "fastText ist nicht installiert. "
            "Installiere bei Bedarf das Paket fasttext und führe diese Zelle erneut aus."
        )
else:
    print("fastText-Baseline wird übersprungen. Setze RUN_FASTTEXT_BASELINE = True, um sie auszuführen.")


fastText-Modell gespeichert: d:\toydev\schwarz-test\models\fasttext\fasttext_politikbereich.bin
fastText-Summary gespeichert: d:\toydev\schwarz-test\models\fasttext\fasttext_summary.csv


,model,CV_Folds,val_accuracy_mean,val_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_weighted_f1_mean,val_weighted_f1_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,fit_time_total_seconds,final_training_time_seconds,model_path,validation_strategy
0,fastText,4,0.889939,0.001607,0.566394,0.001313,0.88376,0.001736,0.548204,0.002851,100.89,31.97,d:\toydev\schwarz-test\models\fasttext\fasttex...,shared_stratified_4_fold_cv


,fold,fit_time_seconds,val_accuracy,val_macro_f1,val_weighted_f1,val_balanced_accuracy
0,1,26.18,0.889286,0.565596,0.882833,0.544169
1,2,26.44,0.891115,0.568033,0.885112,0.549396
2,3,24.21,0.887631,0.564705,0.881393,0.547306
3,4,24.06,0.891725,0.567243,0.885701,0.551946


,precision,recall,f1-score,support
Antidiskriminierung,0.668531,0.708148,0.687770,675.0
Arbeit,0.934086,0.980147,0.956562,6246.0
"Bauen, Wohnen",0.000000,0.000000,0.000000,83.0
Berlin-Image,0.000000,0.000000,0.000000,4.0
Bildung,0.941001,0.934025,0.937500,2698.0
"Bürgerschaftliches Engagement, Bürgerbeteiligung",0.611663,0.656940,0.633493,1405.0
Denkmalschutz,0.603865,0.592417,0.598086,211.0
Europa,1.000000,0.011765,0.023256,85.0
Familie,0.922995,0.857853,0.889232,1006.0
Finanzen,0.000000,0.000000,0.000000,14.0


fastText dient als effizienter NLP-Vergleich zu den klassischen TF-IDF-Modellen. Die Ergebnisse werden auf einem internen Validierungssplit berechnet und sind daher nicht direkt mit der finalen Testevaluation gleichzusetzen. Sie zeigen aber, ob eine kompakte n-Gramm-basierte Textrepräsentation als zusätzliche Modellfamilie konkurrenzfähig sein kann.


### 5.7.5. BERT trainieren, auswerten und speichern

Für BERT wird ebenfalls nur ein interner Validierungssplit aus den Trainingsdaten verwendet. Das Fine-Tuning ist deutlich rechenintensiver als die klassischen Modelle und sollte vorzugsweise in Colab mit GPU ausgeführt werden. Das trainierte Modell, der Tokenizer, das Labelmapping und die Validierungsergebnisse werden gespeichert.


In [3]:
import os
import time
from dotenv import load_dotenv

load_dotenv()

hf_token = os.environ.get("HF_TOKEN")

print("HF_TOKEN vorhanden:", bool(hf_token))
print("RUN_BERT_BASELINE:", RUN_BERT_BASELINE)
print("bert_model_checkpoint:", bert_model_checkpoint)

HF_TOKEN vorhanden: True
RUN_BERT_BASELINE: True
bert_model_checkpoint: deepset/gbert-base


In [ ]:
from dotenv import load_dotenv
load_dotenv()
hf_token = os.environ.get("HF_TOKEN")

if RUN_BERT_BASELINE:
    bert_model_dir = model_output_dir / "bert"
    bert_model_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    bert_train_texts, bert_valid_texts, bert_y_train_raw, bert_y_valid_raw = (
        train_test_split(
            X_train_nlp["model_text"],
            y_train,
            test_size=0.1,
            random_state=42,
            stratify=y_train,
        )
    )

    bert_label_encoder = LabelEncoder()
    bert_label_encoder.fit(y_train)

    bert_y_train = bert_label_encoder.transform(bert_y_train_raw)
    bert_y_valid = bert_label_encoder.transform(bert_y_valid_raw)

    bert_label_mapping = pd.DataFrame(
        {
            "label_id": range(len(bert_label_encoder.classes_)),
            "politikbereich": bert_label_encoder.classes_,
        }
    )

    bert_label_mapping_path = bert_model_dir / "bert_label_mapping.csv"
    bert_label_mapping.to_csv(
        bert_label_mapping_path,
        index=False,
        sep=";",
        encoding="utf-8",
    )

    dump(
        bert_label_encoder,
        bert_model_dir / "bert_label_encoder.joblib",
    )

    try:
        import numpy as np
        import torch
        from transformers import (
            AutoModelForSequenceClassification,
            AutoTokenizer,
            Trainer,
            TrainingArguments,
        )

        bert_start_time = time.perf_counter()

        bert_tokenizer = AutoTokenizer.from_pretrained(
            bert_model_checkpoint,
            token=hf_token,
            use_fast=False,
        )

        train_encodings = bert_tokenizer(
            bert_train_texts.tolist(),
            truncation=True,
            padding=True,
            max_length=256,
        )

        valid_encodings = bert_tokenizer(
            bert_valid_texts.tolist(),
            truncation=True,
            padding=True,
            max_length=256,
        )

        class EncodedPolitikbereichDataset(torch.utils.data.Dataset):
            """Vor-tokenisierter Datensatz für BERT."""

            def __init__(
                self,
                encodings,
                labels,
            ):
                self.encodings = encodings
                self.labels = labels

            def __len__(self):
                return len(self.labels)

            def __getitem__(self, index):
                item = {
                    key: torch.tensor(value[index]) for key, value in self.encodings.items()
                }
                item["labels"] = torch.tensor(
                    self.labels[index],
                    dtype=torch.long,
                )
                return item

        bert_train_dataset = EncodedPolitikbereichDataset(
            train_encodings,
            bert_y_train,
        )
        bert_valid_dataset = EncodedPolitikbereichDataset(
            valid_encodings,
            bert_y_valid,
        )

        bert_model = AutoModelForSequenceClassification.from_pretrained(
            bert_model_checkpoint,
            token=hf_token,
            num_labels=len(bert_label_encoder.classes_),
            id2label={
                index: label for index, label in enumerate(bert_label_encoder.classes_)
            },
            label2id={
                label: index for index, label in enumerate(bert_label_encoder.classes_)
            },
        )

        def compute_bert_metrics(eval_prediction):
            logits, labels = eval_prediction
            predictions = np.argmax(logits, axis=-1)

            return {
                "accuracy": accuracy_score(labels, predictions),
                "macro_f1": f1_score(
                    labels,
                    predictions,
                    average="macro",
                    zero_division=0,
                ),
                "weighted_f1": f1_score(
                    labels,
                    predictions,
                    average="weighted",
                    zero_division=0,
                ),
                "balanced_accuracy": balanced_accuracy_score(
                    labels,
                    predictions,
                ),
            }

        training_arguments_kwargs = {
            "output_dir": str(bert_model_dir / "training_output"),
            "num_train_epochs": 2,
            "learning_rate": 2e-5,
            "per_device_train_batch_size": 8,
            "per_device_eval_batch_size": 16,
            "weight_decay": 0.01,
            "logging_steps": 50,
            "save_strategy": "epoch",
            "load_best_model_at_end": True,
            "metric_for_best_model": "macro_f1",
            "greater_is_better": True,
            "report_to": "none",
        }

        try:
            bert_training_args = TrainingArguments(
                eval_strategy="epoch",
                **training_arguments_kwargs,
            )
        except TypeError:
            bert_training_args = TrainingArguments(
                evaluation_strategy="epoch",
                **training_arguments_kwargs,
            )

        bert_trainer = Trainer(
            model=bert_model,
            args=bert_training_args,
            train_dataset=bert_train_dataset,
            eval_dataset=bert_valid_dataset,
            compute_metrics=compute_bert_metrics,
        )

        bert_trainer.train()
        bert_eval_metrics = bert_trainer.evaluate()

        bert_prediction_output = bert_trainer.predict(bert_valid_dataset)
        bert_y_pred = np.argmax(
            bert_prediction_output.predictions,
            axis=-1,
        )

        bert_training_time = time.perf_counter() - bert_start_time

        bert_saved_model_path = bert_model_dir / "bert_politikbereich"
        bert_trainer.save_model(str(bert_saved_model_path))
        bert_tokenizer.save_pretrained(str(bert_saved_model_path))

        bert_summary = pd.DataFrame(
            [
                {
                    "model": bert_model_checkpoint,
                    "validation_rows": len(bert_y_valid),
                    "accuracy": accuracy_score(
                        bert_y_valid,
                        bert_y_pred,
                    ),
                    "macro_f1": f1_score(
                        bert_y_valid,
                        bert_y_pred,
                        average="macro",
                        zero_division=0,
                    ),
                    "weighted_f1": f1_score(
                        bert_y_valid,
                        bert_y_pred,
                        average="weighted",
                        zero_division=0,
                    ),
                    "balanced_accuracy": balanced_accuracy_score(
                        bert_y_valid,
                        bert_y_pred,
                    ),
                    "training_time_seconds": round(
                        bert_training_time,
                        2,
                    ),
                    "model_path": str(bert_saved_model_path),
                }
            ]
        )

        bert_report = pd.DataFrame(
            classification_report(
                bert_y_valid,
                bert_y_pred,
                target_names=bert_label_encoder.classes_,
                output_dict=True,
                zero_division=0,
            )
        ).transpose()

        bert_summary_path = bert_model_dir / "bert_summary.csv"
        bert_report_path = bert_model_dir / "bert_classification_report.csv"

        bert_summary.to_csv(
            bert_summary_path,
            index=False,
            sep=";",
            encoding="utf-8",
        )
        bert_report.to_csv(
            bert_report_path,
            sep=";",
            encoding="utf-8",
        )

        if "mlflow" in globals():
            with mlflow.start_run(
                run_name="BERT validation",
                nested=True,
            ):
                mlflow.log_param("model", bert_model_checkpoint)
                mlflow.log_param("num_train_epochs", 2)
                mlflow.log_param("learning_rate", 2e-5)
                mlflow.log_param("max_length", 256)
                for metric_name, metric_value in bert_summary.iloc[0].items():
                    if isinstance(metric_value, (int, float)):
                        mlflow.log_metric(metric_name, metric_value)
                mlflow.log_artifact(str(bert_summary_path))
                mlflow.log_artifact(str(bert_report_path))
                mlflow.log_artifact(str(bert_label_mapping_path))

        print("BERT-Modell gespeichert:", bert_saved_model_path)
        print("BERT-Summary gespeichert:", bert_summary_path)
        display(bert_summary)
        display(bert_report.head(10))

    except ImportError:
        print(
            "transformers und/oder torch sind nicht installiert. "
            "Installiere diese Pakete und führe die Zelle erneut aus."
        )
else:
    print("BERT-Baseline wird ?bersprungen. Setze RUN_BERT_BASELINE = True, um sie auszuf?hren.")


ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.